# Labor 3: Reglersynthese

In [ ]:
import control as ctrl
import matplotlib.pyplot as plt
import numpy as np

## Aufgabe 1: P-Regler

$$G(s) = \frac{1}{s^2 + 2s + 1} = \frac{1}{(s+1)(s+1)}$$

(a)
Charakteristische Gleichung des geregelten Systems
$$1+GK = 0$$
$$s^2 + 2s + 1 + k_P = 0$$

(b)
Wunschpole $p_1$ und $p_2$ des geschlossenen Kreises
$$(s-p_1) (s-p_2) = 0$$
$$s^2 -(p_1+p_2) s + p1 p2 = 0$$

Koeffizientenvergleich:
* Koeffizient von $s^2$ ist jeweils 1 (kommt durch die Normierung)
* Koeffizient von $s = s^1$ ist einmal 2, wir wollen aber $-(p_1+p_2)$
* Koeffizient von $1 = s^0$ ist $1+k_P$, wir wollen $p_1 p_2$

Der Daempfungsterm (bzw. der Koeffizient vor $s=s^1$) kann durch den P-Regler *nicht* beeinflusst werden (fuer *diese* Strecke).
Die Pole koennen also *nicht beliebig* platziert werden.
Wir koennen dem geschlossenen Kreis keine beliebige Dynamik aufpraegen.

(c)
Sprungantwort und Pol-Nullstellen-Diagramm des geschlossenen Kreises.

In [ ]:
t_eval = np.linspace(0,10,1000)

s = ctrl.tf('s')
G = 1/(s**2+2*s+1)

t, h = ctrl.step_response(G, T=t_eval)

plt.figure('Sprungantworten')
plt.clf()
plt.plot(t, h, label='Ungeregeltes System')


# Perfekter Sensor: Übertragungsfunktion = 1
G_sensor = ctrl.tf([1], [1])
    
for k_P in [1,2,5,10,100]:
    K = ctrl.tf([k_P], [1])
    
    G_sys = ctrl.feedback(ctrl.series(K, G), G_sensor)

    t, h_sys = ctrl.step_response(G_sys, T=t_eval)
    plt.plot(t, h_sys, label=f'k_P={k_P}')

plt.legend()
plt.grid(True)
plt.xlabel('t')
plt.ylabel('h(t)')
plt.show()

In [ ]:
p = ctrl.poles(G)
z = ctrl.zeros(G)

plt.figure('Pol-(Nullstellen-)Diagramm') # es gibt hier keine Nullstellen
plt.clf()
plt.scatter(np.real(p), np.imag(p), label='Ungeregeltes System')

for k_P in [1,2,5,10,100]:
    K = ctrl.tf([k_P], [1])
    G_sys = ctrl.feedback(ctrl.series(K, G), G_sensor)

    p = ctrl.poles(G_sys)
    z = ctrl.zeros(G_sys)
    plt.scatter(np.real(p), np.imag(p), label=f'k_P={k_P}')

plt.legend()
plt.grid(True)
plt.xlabel('Re(s)')
plt.ylabel('Im(s)')
plt.xlim([-2,2])
plt.ylim([-12,12])
plt.show()

Zu (c):
Es fällt auf, dass eine Erhöhung von $k_P$ zwar die bleibende Regelabweichung reduziert, jedoch auf Kosten eines stärkeren Überschwingens.

Im Pol-(Nullstellen-)Diagramm erkennt man dass die Polstellen auf einer Geraden parallel zur imaginären Achse liegen. Eine Erhöhung von $k_P$ führt zu einer Erhöhung von $\omega_n$ (daher die kürzere Anstiegszeit), gleichzeit jedoch auch zu einer Erhöhung von $\zeta$ (daher das stärkere Überschwingen). Die Einschwingzeit bleibt konstant.

Teaser: Sobald wir die *Wurzelortskurve* (Kap. 4) kennen und verstehen, werden wir nahezu direkt sehen, wo die Polstellen des geschlossenen Kreises liegen und wie sie sich bei einer Erhöhung von $k_P$ verschieben.

## Aufgabe 2: Robustheit

$$G(s) = \frac{s+1}{s-a}$$

(a)
$$K(s) = G^{-1}(s) = \frac{s-a}{s+1}$$

Daraus folgt
$$GK = \frac{s+1}{s-a} \cdot \frac{s-a}{s+1} = \frac{s-a}{s-a}$$

Mit Kürzen des Pols bei $s=a$
$$GK = 1$$

(b)
Es dürfen nur stabile Pole gekürzt werden. Instabile Pole dürfen nicht gekürzt werden.

* $a<0$ ist ok für eine Steuerung
* $a>0$ keine Steuerung möglich


In [ ]:
# Aufgabenteile (c) und (d): Steuerung
# Oder: Was passiert beim kuerzen instabiler Pole?

s = ctrl.tf('s')
# Parameter der Strecke
a = 1.0

# Aufgabenteil (c): a sei exakt bekannt
a_K = a

# Aufgabenteil (d)
# Approximation des Streckenparameters 'a' in der Steuerung
a_K = 0.99*a

K = (s-a_K)/(s+1)
G = (s+1)/(s-a)
GK = ctrl.series(K, G)

# Pole ausgeben: Man sieht: Der Pol bei a = 1.0 besteht weiterhin.
print('Pole der offenen Steuerkette:')
print(ctrl.poles(GK))

t_eval = np.linspace(0,10,100)
t, h = ctrl.step_response(GK, T=t_eval)
plt.figure(1)
plt.clf()
plt.plot(t, h)
plt.grid(True)
plt.xlabel('t')
plt.ylabel('y(t)')
plt.show()

### Regelung

In [ ]:
# Aufgabenteile (e)-(g): Regelung

# Aufgabenteil (f): a sei exakt bekannt
a = 1
a_K = a
print(f'a={a}')

# Aufgabenteil (g)
# Approximation des Streckenparameters 'a' im Regler
a_K = 0.99*a

# P-Regler
p = -3
k_P = -(p-a_K)/(p+1)

print(f'k_P = {k_P} (fuer a = {a})')

K = k_P
G = (s+1)/(s-a)
GK = ctrl.series(K, G)
G_cl = ctrl.feedback(GK, ctrl.tf([1],[1]))

print('Pole des geschlossenen Kreises:')
print(ctrl.poles(G_cl))


t_eval = np.linspace(0,100,1000)
t, h = ctrl.step_response(G_cl, T=t_eval)
plt.figure(1)
plt.clf()
plt.plot(t, h)
plt.grid(True)
plt.xlabel('t')
plt.ylabel('y(t)')
plt.show()


## Aufgabe 3: Bleibende Regelabweichung

In [ ]:
s = ctrl.tf('s')
G = 1/s
K = 2*(s+1)/s
H = 100/(s+100)
# H = 1

# Uebertragung von W auf Y (Annahme: Z=0)
G_WY = ctrl.feedback(ctrl.series(K, G), H)

# Uebertragung von Z auf Y (Annahme: W=0)
G_ZY = ctrl.feedback(G, ctrl.series(H, K))

print(f'G_WY = {G_WY}')
print(f'G_ZY = {G_ZY}')

In [ ]:
# Folgeregelungsfehler bei W = Einheitssprung und Z = 0
t_eval = np.linspace(0,30,3000)
t, h = ctrl.step_response(G_WY, T=t_eval)

plt.figure(3)
plt.clf()
plt.plot(t, h, label='y(t) (Regelgroesse)')
plt.plot([0, 0, 10], [0, 1, 1], label='w(t) (Fuehrungsgroesse)')
plt.grid(True)
plt.xlabel('t')
plt.ylabel('w(t), y(t)')
plt.legend()
plt.title('Fuehrungsgroesse: Einheitssprung')
plt.show()

In [ ]:
# Folgeregelungsfehler bei W = Rampe und Z = 0
# Um die Rampe zu erzeugen, schalten wir einen Integrator vor das
# Gesamtsystem.
# w_urspruenglich --> (1/s) --> w_integriert
# Sprung          --> (1/s) --> Rampe
G_integrator = 1/s
t, y = ctrl.step_response(ctrl.series(G_integrator, G_WY), T=t_eval)


# Rampe als Fuehrungsgroesse
w = t
# Regelfehler (e = w - y)
e = w - y

plt.figure(4)
plt.clf()
plt.plot(t, y, label='y(t) (Regelgroesse)')
plt.plot(t, w, label='w(t) (Fuehrungsgroesse)')
plt.plot(t, 100*e, label='e(t) (Regelfehler) * 100')
plt.grid(True)
plt.xlabel('t')
plt.ylabel('w(t), y(t)')
plt.legend()
plt.title('Fuehrungsgroesse: Rampe')
plt.ylim([-5,35])
plt.show()

print(e[-5:])

## Aufgabe 7: Contourplots

In [ ]:
# %% Draw a heatmap

# Raum R^2
X, Y = np.mgrid[-5:5:0.02, -5:5:0.02]
Z = np.zeros_like(X)

# Raum C
S = X + 1j*Y

def G_fun(s):
    val = 1.0 / s
    # val = s
    s1 = 2+3j
    s2 = 2-3j
    s3 = -3
    val = (s-s3)/((s-s1)*(s-s2))
    return val

G = G_fun(S)

# Z = np.abs(G)
Z = np.rad2deg(np.angle(G))

plt.figure(1)
plt.clf()
plt.contourf(X, Y, Z, levels=64)
plt.colorbar()
plt.show()

# Aufgabe 8: Der Effekt von Nullstellen

In [ ]:
omega_n = 1.0
zeta = 0.5
alpha = -10

G_no_zero = ctrl.tf([1], [1/omega_n**2, 2*zeta/omega_n, 1])

G = ctrl.tf([1/(alpha*zeta*omega_n), 1], [1/omega_n**2, 2*zeta/omega_n, 1])
print(G)

t_no_zero, y_no_zero = ctrl.step_response(G_no_zero)
t, y = ctrl.step_response(G)

fig = plt.figure('Step response')
plt.clf()
plt.plot(t,y, label='with zero')
plt.plot(t_no_zero, y_no_zero, linestyle='dashed', label='no zero')
plt.grid(True)
plt.legend()
plt.show()

plt.figure('Pol-Nullstellen')
ctrl.pzmap(G)